# Verificación de locutor de texto dependiente

Proyecto de Biometría de la Voz. El objetivo es construir un sistema que, dado un
audio de una persona diciendo una frase corta predefinida, sea capaz de:

1. **Identificar al locutor** — ¿quién está hablando? (Red 1)
2. **Verificar la frase** — ¿ha dicho la frase esperada? (Red 2)

Este notebook es el *orquestador*: no contiene la lógica interna, solo llama en
orden a las funciones de los módulos `src/data.py`, `src/model.py` y
`src/train.py`, y muestra los resultados.

## Base de datos

Usamos `TextDependentSpeakerIdentification`: 816 locutores que pronuncian 5
frases cortas predefinidas (S1–S5), unos 45.000 audios en total a 16 kHz.

Por viabilidad computacional (RTX 3060 Laptop, 6 GB) trabajamos con un
**subconjunto de los primeros 50 locutores** (orden alfabético). Esto nos deja
unos **2.500 audios**, ~10 por locutor y frase, repartidos en:

- **50 clases de locutor** para la Red 1.
- **5 clases de frase** (S1–S5) para la Red 2.

## 1. Preparación del entorno y carga de datos

Importamos los módulos del proyecto. La línea `sys.path.append("src")` le dice a
Python que busque también en la carpeta `src/`, que es donde están nuestros
módulos.

A continuación cargamos el dataset. Procesar los ~2.500 audios (leer cada WAV
y extraer sus MFCC) tarda un par de minutos, así que usamos una **caché**: la
primera vez se procesa todo y se guarda en un fichero `cache_dataset.npz`; las
siguientes veces se carga directamente de ahí, en segundos.

> Si cambias el subconjunto de locutores (`max_locutores`), borra antes
> `cache_dataset.npz` para forzar la regeneración.

In [4]:
import sys, os
sys.path.append("src")

import numpy as np
import data, model, train

CACHE = "cache_dataset.npz"
N_LOCUTORES = 50   # subconjunto de trabajo

if os.path.exists(CACHE):
    # La caché ya existe: cargamos de disco (rápido)
    d = np.load(CACHE, allow_pickle=True)
    X, y_loc, y_pin = d["X"], d["y_loc"], d["y_pin"]
    mapas     = d["mapas"].item()
    registros = d["registros"].tolist()
    print("Dataset cargado de la caché:", X.shape)
else:
    # Primera vez: procesamos los audios y guardamos la caché
    registros = data.construir_indice(max_locutores=N_LOCUTORES)
    X, y_loc, y_pin, mapas = data.construir_dataset(registros)
    np.savez(CACHE, X=X, y_loc=y_loc, y_pin=y_pin,
             mapas=mapas, registros=registros)
    print("Dataset procesado y guardado en caché:", X.shape)

Índice construido: 2528 audios
  Locutores : 50
  Frases    : 5
  Audios/locutor: 50 media
  Procesando audio 0/2528...
  Procesando audio 250/2528...
  Procesando audio 500/2528...
  Procesando audio 750/2528...
  Procesando audio 1000/2528...
  Procesando audio 1250/2528...
  Procesando audio 1500/2528...
  Procesando audio 1750/2528...
  Procesando audio 2000/2528...
  Procesando audio 2250/2528...
  Procesando audio 2500/2528...

Dataset construido:
  X         : (2528, 120, 63, 1)
  y_locutor : (2528,)  (50 clases)
  y_pin     : (2528,)  (5 clases)
Dataset procesado y guardado en caché: (2528, 120, 63, 1)


In [5]:
# Comprobación de coherencia: que la caché coincide con lo esperado
print(f"Audios totales : {X.shape[0]}")
print(f"Forma MFCC     : {X.shape[1:]}  (n_caract, T, canal)")
print(f"Locutores      : {len(mapas['locutor'])}  (esperados: {N_LOCUTORES})")
print(f"Frases         : {len(mapas['pin'])}      (esperadas: 5)")

assert len(mapas["locutor"]) == N_LOCUTORES, \
    "La caché no coincide con N_LOCUTORES. ¿Has olvidado borrar cache_dataset.npz?"
assert len(mapas["pin"]) == 5, "Esperábamos 5 frases (S1–S5)."

Audios totales : 2528
Forma MFCC     : (120, 63, 1)  (n_caract, T, canal)
Locutores      : 50  (esperados: 50)
Frases         : 5      (esperadas: 5)
